In [1]:
# Test Environment
import ee
import geemap
import pygadm
import geopandas as gpd
import geojson

In [2]:
# Connect to GEE
ee.Initialize(project="tutorial-geoscripting-paulian")

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


## 1 IMAGE SNAPSHOT MAP
cells 3-5 

In [3]:
# Import image
s1_collection = ee.ImageCollection("COPERNICUS/S1_GRD")

In [4]:
# set our initial zoom position for Constanta, Romania
center_lat = 44.1092
center_lon = 28.7499
zoomlevel = 12

# The following line will initialize the map
Map = geemap.Map(center=[center_lat, center_lon], zoom=zoomlevel)

# The following line will actually call and show the map:
Map

Map(center=[44.1092, 28.7499], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topri…

In [ ]:
# Small time interval to get 1 snapshot
from_date = '2016-07-20'
to_date = '2026-07-21'

# Constanța port plus the offshore anchorage zone
bbox = [28.64, 44.08, 28.85, 44.18]
aoi = ee.Geometry.Rectangle(bbox)

# Image filter
image = s1_collection\
            .filterBounds(aoi)\
            .filterDate(from_date,to_date)\
            .sort("system:time_start", True)\
            .first()

# VV Polarization
vizualisation_params = {
    'min': -30,
    'max': -5,
    'opacity':1,
    'bands':['VH']
}

# Add image to base map
Map.addLayer(ee_object = image.select('VH').clip(aoi),
             vis_params = vizualisation_params,
             name = "first image")

In [ ]:
print(image.date().format('YYYY-MM-dd HH:mm').getInfo())

## PROCESS SAR

In [ ]:
from functions.detection_algo import load_water
water = load_water('images/port_osm/clip_water.shp', bbox) #shapefile source: "https://osmdata.openstreetmap.de/data/water-polygons.html"

In [8]:
from functions.detection_algo import detect_ships

ships = detect_ships(image, water)
print(ships.size().getInfo())                   # vessel count
Map.addLayer(water, {'color': 'blue'}, 'water mask')
Map.addLayer(ships, {'color': 'red'}, 'ships')

176


## 2019-2026 SAR

In [ ]:
start_date = '2019-01-01'
end_date   = '2026-01-01'   # end is exclusive, so this includes all of 2025

s1 = (ee.ImageCollection('COPERNICUS/S1_GRD')
      .filterBounds(aoi)
      .filterDate(start_date, end_date)
      .filter(ee.Filter.eq('instrumentMode', 'IW'))
      .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
      .sort('system:time_start'))

# Which relative orbits cover the AOI, and how many scenes each
print(s1.aggregate_histogram('relativeOrbitNumber_start').getInfo())

### Specify Most Frequent Orbit

In [ ]:
ORBIT = 29   # replace with the most frequent one from the histogram
s1 = s1.filter(ee.Filter.eq('relativeOrbitNumber_start', ORBIT))

print(s1.size().getInfo())
dates = s1.aggregate_array('system:time_start') \
          .map(lambda t: ee.Date(t).format('YYYY-MM-dd HH:mm')).getInfo()
print(dates[:5], dates[-5:])

### GET SAR COVERING FULL AOI

In [ ]:
# Option A) Keep only scenes that fully contain the AOI
s1_full = s1.filter(ee.Filter.contains(leftField='.geo', rightValue=aoi))
print(s1.size().getInfo(), s1_full.size().getInfo())

In [ ]:
# Option B) Mosaic the slices from the same pass
def daily_mosaic(d):
    d = ee.Date(d)
    day = s1.filterDate(d, d.advance(1, 'day'))
    return day.mosaic().clip(aoi).set('system:time_start', d.millis())

days = s1.aggregate_array('system:time_start') \
         .map(lambda t: ee.Date(t).format('YYYY-MM-dd')).distinct()
s1_daily = ee.ImageCollection(days.map(daily_mosaic))

### PROCESS SAR 

In [ ]:
from functions.detection_algo import s1_detect_ships, ship_counts

water = load_water('images/port_osm/clip_water.shp', bbox)

# time series
df = ship_counts(s1_daily, water)
df.to_csv('data/ship_counts.csv')   # save once, so you don't rerun GEE every time

# single date for the map
polys = s1_detect_ships(s1_daily.first(), water, vectors=True)